In [ ]:
!pip install -q gdown

import gdown

file_id = "1K56kVYHHDfLA2Anm7ga0tQolMwIPk6R8"
url = f"https://drive.google.com/uc?id={file_id}"

output = "dataset.tar"

gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1K56kVYHHDfLA2Anm7ga0tQolMwIPk6R8
From (redirected): https://drive.google.com/uc?id=1K56kVYHHDfLA2Anm7ga0tQolMwIPk6R8&confirm=t&uuid=07d577a0-52f3-4bc5-abf1-6932498a4897
To: /content/dataset.tar
100%|██████████| 37.9G/37.9G [06:19<00:00, 99.8MB/s]


'dataset.tar'

In [ ]:
from __future__ import annotations

import csv
import json
import random
import re
import shutil
import tarfile
import zipfile
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable


IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
IDENTITY_RE = re.compile(r"^n\d+$", re.IGNORECASE)


@dataclass(frozen=True)
class PlannedFile:
    split: str
    dataset_label: int
    identity_id: str
    identity_name: str
    source_path: str
    image_path: Path


def is_image_member(member_name: str) -> bool:
    return Path(member_name).suffix.lower() in IMAGE_EXTENSIONS


def parse_identity_id(member_name: str) -> str | None:
    """
    Cerca la cartella identità dentro il percorso dell'archivio.
    Funziona sia con VGGFace2 tipo n000001/... sia con dataset organizzati
    come nome_classe/immagine.jpg.
    """
    parts = Path(member_name.replace("\\", "/")).parts

    for part in parts:
        if IDENTITY_RE.match(part):
            return part

    if len(parts) >= 2:
        return parts[-2]

    return None


def safe_dir_name(dataset_label: int, identity_id: str, identity_name: str) -> str:
    clean_id = re.sub(r"[^A-Za-z0-9_.-]+", "_", identity_id).strip("_")
    clean_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", identity_name).strip("_")
    return f"{dataset_label:03d}_{clean_id}_{clean_name}"


def load_identity_metadata(identity_meta_path: Path | None) -> dict[str, str]:
    if identity_meta_path is None:
        return {}

    raw_path = str(identity_meta_path).strip()
    if not raw_path:
        return {}

    if not identity_meta_path.exists():
        raise FileNotFoundError(f"identity metadata not found: {identity_meta_path}")

    with identity_meta_path.open("r", encoding="utf-8-sig", newline="") as handle:
        sample = handle.read(4096)
        handle.seek(0)
        has_header = "Class_ID" in sample and "Name" in sample

        if has_header:
            reader = csv.DictReader(handle)
            rows = {}

            for row in reader:
                identity_id = (row.get("Class_ID") or row.get("class_id") or "").strip()
                identity_name = (row.get("Name") or row.get("name") or "").strip()

                if identity_id and identity_name:
                    rows[identity_id] = identity_name

            return rows

        reader = csv.reader(handle)
        rows = {}

        for row in reader:
            if len(row) >= 2:
                identity_id = row[0].strip()
                identity_name = row[1].strip()

                if identity_id and identity_name:
                    rows[identity_id] = identity_name

        return rows


def reservoir_scan_archive(
    archive_path: Path,
    required_per_identity: int,
    rng: random.Random,
) -> tuple[dict[str, list[str]], dict[str, int]]:
    """
    Scansiona il .tar senza estrarlo tutto.
    Tiene solo un campione casuale di immagini per ogni identità.
    """
    samples_by_identity: dict[str, list[str]] = defaultdict(list)
    counts_by_identity: dict[str, int] = defaultdict(int)

    with tarfile.open(archive_path, mode="r|*") as tar:
        for member_index, member in enumerate(tar, start=1):
            if member_index % 100000 == 0:
                print(f"Scansionati {member_index:,} elementi dell'archivio...")

            if not member.isfile() or not is_image_member(member.name):
                continue

            identity_id = parse_identity_id(member.name)

            if identity_id is None:
                continue

            counts_by_identity[identity_id] += 1
            seen = counts_by_identity[identity_id]
            reservoir = samples_by_identity[identity_id]

            if len(reservoir) < required_per_identity:
                reservoir.append(member.name)
            else:
                replace_index = rng.randrange(seen)

                if replace_index < required_per_identity:
                    reservoir[replace_index] = member.name

    return dict(samples_by_identity), dict(counts_by_identity)


def select_identities(
    samples_by_identity: dict[str, list[str]],
    counts_by_identity: dict[str, int],
    num_identities: int,
    required_per_identity: int,
    rng: random.Random,
) -> list[str]:
    candidates = [
        identity_id
        for identity_id, count in counts_by_identity.items()
        if count >= required_per_identity
        and len(samples_by_identity.get(identity_id, [])) >= required_per_identity
    ]

    rng.shuffle(candidates)

    if len(candidates) < num_identities:
        raise RuntimeError(
            f"Solo {len(candidates)} identità hanno almeno "
            f"{required_per_identity} immagini, ma ne hai richieste {num_identities}."
        )

    return candidates[:num_identities]


def create_extraction_plan(
    selected_identities: list[str],
    samples_by_identity: dict[str, list[str]],
    counts_by_identity: dict[str, int],
    identity_metadata: dict[str, str],
    output_root: Path,
    train_per_identity: int,
    test_per_identity: int,
    rng: random.Random,
) -> tuple[dict[str, PlannedFile], list[dict[str, object]]]:
    plan: dict[str, PlannedFile] = {}
    identity_rows: list[dict[str, object]] = []

    for dataset_label, identity_id in enumerate(selected_identities):
        identity_name = identity_metadata.get(identity_id, identity_id)
        identity_dir = safe_dir_name(dataset_label, identity_id, identity_name)

        samples = list(samples_by_identity[identity_id])
        rng.shuffle(samples)

        split_samples = {
            "train": samples[:train_per_identity],
            "test": samples[
                train_per_identity : train_per_identity + test_per_identity
            ],
        }

        for split, member_names in split_samples.items():
            for sample_index, member_name in enumerate(member_names):
                extension = Path(member_name).suffix.lower()

                destination_path = (
                    output_root
                    / split
                    / identity_dir
                    / f"{sample_index:04d}{extension}"
                )

                plan[member_name] = PlannedFile(
                    split=split,
                    dataset_label=dataset_label,
                    identity_id=identity_id,
                    identity_name=identity_name,
                    source_path=member_name,
                    image_path=destination_path,
                )

        identity_rows.append(
            {
                "dataset_label": dataset_label,
                "identity_id": identity_id,
                "identity_name": identity_name,
                "available_images": counts_by_identity[identity_id],
            }
        )

    return plan, identity_rows


def extract_planned_files(archive_path: Path, plan: dict[str, PlannedFile]) -> None:
    remaining = set(plan)
    total = len(remaining)

    with tarfile.open(archive_path, mode="r|*") as tar:
        for member_index, member in enumerate(tar, start=1):
            if member_index % 100000 == 0:
                print(
                    f"Estrazione: scansionati {member_index:,} elementi, "
                    f"rimangono {len(remaining):,}/{total:,} file..."
                )

            planned = plan.get(member.name)

            if planned is None:
                continue

            source = tar.extractfile(member)

            if source is None:
                continue

            planned.image_path.parent.mkdir(parents=True, exist_ok=True)

            with planned.image_path.open("wb") as destination:
                shutil.copyfileobj(source, destination)

            remaining.remove(member.name)

            if not remaining:
                break

    if remaining:
        missing_preview = sorted(remaining)[:5]

        raise RuntimeError(
            f"{len(remaining)} file pianificati non sono stati estratti. "
            f"Esempi: {missing_preview}"
        )


def write_csv(path: Path, fieldnames: list[str], rows: Iterable[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def write_outputs(
    output_root: Path,
    plan: dict[str, PlannedFile],
    identity_rows: list[dict[str, object]],
    summary: dict[str, object],
) -> None:
    train_rows: list[dict[str, object]] = []
    test_rows: list[dict[str, object]] = []

    for planned in sorted(plan.values(), key=lambda item: str(item.image_path)):
        row = {
            "split": planned.split,
            "dataset_label": planned.dataset_label,
            "identity_id": planned.identity_id,
            "identity_name": planned.identity_name,
            "source_path": planned.source_path,
            "image_path": str(planned.image_path),
        }

        if planned.split == "train":
            train_rows.append(row)
        elif planned.split == "test":
            test_rows.append(row)

    split_root = output_root / "splits"

    fieldnames = [
        "split",
        "dataset_label",
        "identity_id",
        "identity_name",
        "source_path",
        "image_path",
    ]

    write_csv(split_root / "train.csv", fieldnames, train_rows)
    write_csv(split_root / "test.csv", fieldnames, test_rows)

    write_csv(
        split_root / "identities.csv",
        ["dataset_label", "identity_id", "identity_name", "available_images"],
        identity_rows,
    )

    (split_root / "summary.json").write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8",
    )


def zip_directory(source_dir: Path, zip_path: Path) -> None:
    zip_path.parent.mkdir(parents=True, exist_ok=True)

    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for file_path in source_dir.rglob("*"):
            if file_path.is_file():
                archive.write(file_path, file_path.relative_to(source_dir.parent))


def create_dataset_from_tar(
    archive_path: str | Path,
    output_root: str | Path = "/content/vggface2_subset",
    identity_meta_path: str | Path | None = None,
    num_identities: int = 100,
    train_per_identity: int = 20,
    test_per_identity: int = 10,
    seed: int = 42,
    zip_output: str | Path | None = "/content/vggface2_subset.zip",
    overwrite: bool = True,
) -> dict[str, object]:

    archive_path = Path(archive_path)
    output_root = Path(output_root)
    identity_meta_path = Path(identity_meta_path) if identity_meta_path else None
    zip_output = Path(zip_output) if zip_output else None

    if not archive_path.exists():
        raise FileNotFoundError(f"Archivio non trovato: {archive_path}")

    if num_identities <= 0:
        raise ValueError("num_identities deve essere positivo")

    if train_per_identity < 0 or test_per_identity <= 0:
        raise ValueError("train_per_identity deve essere >= 0 e test_per_identity > 0")

    required_per_identity = train_per_identity + test_per_identity
    rng = random.Random(seed)

    if output_root.exists() and overwrite:
        shutil.rmtree(output_root)

    output_root.mkdir(parents=True, exist_ok=True)

    print("Caricamento metadata opzionali...")
    identity_metadata = load_identity_metadata(identity_meta_path)

    print("Passo 1/2: scansione archivio e campionamento immagini...")
    samples_by_identity, counts_by_identity = reservoir_scan_archive(
        archive_path,
        required_per_identity,
        rng,
    )

    print("Identità trovate:", len(counts_by_identity))

    selected_identities = select_identities(
        samples_by_identity,
        counts_by_identity,
        num_identities,
        required_per_identity,
        rng,
    )

    print(f"Identità selezionate: {len(selected_identities)}")

    plan, identity_rows = create_extraction_plan(
        selected_identities=selected_identities,
        samples_by_identity=samples_by_identity,
        counts_by_identity=counts_by_identity,
        identity_metadata=identity_metadata,
        output_root=output_root,
        train_per_identity=train_per_identity,
        test_per_identity=test_per_identity,
        rng=rng,
    )

    print(f"Passo 2/2: estrazione di {len(plan)} immagini selezionate...")
    extract_planned_files(archive_path, plan)

    summary = {
        "archive": str(archive_path),
        "identity_meta": str(identity_meta_path) if identity_meta_path else None,
        "output_root": str(output_root),
        "num_identities": num_identities,
        "train_per_identity": train_per_identity,
        "test_per_identity": test_per_identity,
        "train_images": num_identities * train_per_identity,
        "test_images": num_identities * test_per_identity,
        "seed": seed,
        "preprocessing": "none; original images extracted unchanged",
    }

    write_outputs(output_root, plan, identity_rows, summary)

    if zip_output is not None:
        print(f"Creazione archivio zip: {zip_output}")
        zip_directory(output_root, zip_output)

    print("Dataset creato correttamente.")
    print(json.dumps(summary, indent=2))

    return summary

In [ ]:
summary = create_dataset_from_tar(
    archive_path="/content/dataset.tar",
    output_root="/content/vggface2_subset",

    # Numero di classi/identità da estrarre
    num_identities=100,

    # Numero immagini per ogni identità
    train_per_identity=20,
    test_per_identity=10,

    # Seed per rendere il campionamento ripetibile
    seed=42,

    # Crea anche uno zip finale scaricabile
    zip_output="/content/vggface2_subset.zip",

    # Cancella output precedente se esiste
    overwrite=True,
)

Caricamento metadata opzionali...
Passo 1/2: scansione archivio e campionamento immagini...
Scansionati 100,000 elementi dell'archivio...
Scansionati 200,000 elementi dell'archivio...
Scansionati 300,000 elementi dell'archivio...
Scansionati 400,000 elementi dell'archivio...
Scansionati 500,000 elementi dell'archivio...
Scansionati 600,000 elementi dell'archivio...
Scansionati 700,000 elementi dell'archivio...
Scansionati 800,000 elementi dell'archivio...
Scansionati 900,000 elementi dell'archivio...
Scansionati 1,000,000 elementi dell'archivio...
Scansionati 1,100,000 elementi dell'archivio...
Scansionati 1,200,000 elementi dell'archivio...
Scansionati 1,300,000 elementi dell'archivio...
Scansionati 1,400,000 elementi dell'archivio...
Scansionati 1,500,000 elementi dell'archivio...
Scansionati 1,600,000 elementi dell'archivio...
Scansionati 1,700,000 elementi dell'archivio...
Scansionati 1,800,000 elementi dell'archivio...
Scansionati 1,900,000 elementi dell'archivio...
Scansionati 2,